In [7]:
# --- imports ---
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["XLU", "FUTY", "VPU"]
scalar_list =  [4.3274, 
                3.3520, 
               # 1.7105 ,
                1.0000 ,
                 ]

# moving_avg_short = 9
# moving_avg_long = 21

filename = f"{'_'.join(sorted_symbols_list)}.csv"

base_directory = Path("..").resolve()

input_path = base_directory / "backtests" / filename
output_path = base_directory / "backtests" / filename


df = pd.read_csv(input_path, index_col=0)

anchor = sorted_symbols_list[-1]

cols_list = []

for i in [0, 1, 2]:
    symbol = sorted_symbols_list[i]
    scalar = scalar_list[i]

    '''
    price_ratio_column = f"to {anchor} / to {symbol}"
    moving_average_column = f"{anchor} / {symbol} moving avg"
    prior_average_column = f"yday {moving_average_column}"

    df[moving_average_column] = (
        df[price_ratio_column].rolling(moving_avg_days).mean()
    )
    df[prior_average_column] = df[moving_average_column].shift(1)
    '''
    df[f'{anchor} / {symbol}'] = df[f'eop {anchor} price'] / df[f'eop {symbol} price']
    df[f'{symbol} scalar'] = scalar
    df[f"eop {symbol} unit price"] = (
        scalar * df[f"eop {symbol} price"]
    )
    cols_list.append(f"eop {symbol} unit price")


'''
for symbol in sorted_symbols_list:
    df[f"{symbol} unit price pct diff"] = np.log(
        df[f"eop {symbol} unit price"] / df[f"eop {anchor} unit price"]
    )

non_anchor = sorted_symbols_list[0]
df["unit price pct diff"] = (
    df[f"{non_anchor} unit price pct diff"]
    - df[f"{anchor} unit price pct diff"]
)

anchor = sorted_symbols_list[1]
non_anchor = sorted_symbols_list[0]
bop_col_name = f'bop {non_anchor}* price'
eop_col_name = bop_col_name.replace('bop', 'eop')

df[bop_col_name] = 0
df[eop_col_name] = df[f"eop {non_anchor} price"] * scalar
df[bop_col_name] = df[eop_col_name].shift(1)

df[f'bop {non_anchor}* - {anchor}'] = df[bop_col_name] - df[f"bop {anchor} price"]
df[f'eop {non_anchor}* - {anchor}'] = df[eop_col_name] - df[f"eop {anchor} price"]

df[f'eop {moving_avg_short} edma'] = (df[f'eop {non_anchor}* - {anchor}']
    .ewm(span=moving_avg_short, min_periods=moving_avg_short).mean())                                      
df[f"eop {moving_avg_long} edma"] = (df[f"eop {non_anchor}* - {anchor}"]
    .ewm(span=moving_avg_long, min_periods=moving_avg_long).mean())
df['edma diff'] = df[f'eop {moving_avg_short} edma'] - df[f"eop {moving_avg_long} edma"]
'''

df["max_price"] = df[cols_list].max(axis=1)
df["min_price"] = df[cols_list].min(axis=1)
df["pct_diff"] = np.log(df['max_price'] / df['min_price'])

df.to_csv(output_path)
print(f"Saved {output_path}")
print("finished")


Saved C:\Users\micha\git-projects\backtesting\backtests\XLU_FUTY_VPU.csv
finished
